# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides an exploratory guide for loading and analyzing the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset via Croissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {getattr(metadata, 'name', None)}\nDescription: {getattr(metadata, 'description', None)}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Here, we inspect the structure of the dataset to discover the available record sets, their fields (columns), and the fields' identifiers. This information is essential for referencing data using their `@id` as required.

In [ ]:
# List available Record Sets and their structure
print('Available Record Sets:')
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"  - Name: {rs.name}, @id: {rs.id}")
    print("    Fields:")
    for field in rs.fields:
        print(f"      - {field.name}  (@id: {field.id}, type: {field.data_type})")
    print('')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Below, we choose the main record set (here, `cr:RecordSet:survey`) and extract all records, loading them into a pandas DataFrame. The DataFrame columns correspond to field `@id`s, following the Croissant specification.

In [ ]:
# Extract data from each record set using their @id
# Note: Replace these IDs with the ones printed in the overview above if they differ
main_record_set_id = 'cr:RecordSet:survey'  # Typical convention for the main survey table
demographics_record_set_id = 'cr:RecordSet:demographics'  # If such a set exists

# List all available @id's found
print('Available record set @id values:')
for rs in record_sets:
    print(f"- {rs.id}")

# Load main record set data
dfs = {}
target_record_sets = [rs.id for rs in record_sets]  # Extract all for flexibility
for rsid in target_record_sets:
    records = list(dataset.records(record_set=rsid))
    if records:
        df = pd.DataFrame(records)
        dfs[rsid] = df
        print(f"Loaded {len(df)} records from {rsid}")
    else:
        print(f"No records found for {rsid}")

# Choose main table for further analysis
primary_table_id = target_record_sets[0]  # Use the first record set by default
print(f"Primary DataFrame columns for {primary_table_id}:\n", dfs[primary_table_id].columns.tolist())
dfs[primary_table_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's experiment with the GAD-7 total score field, referenced by its `@id` (e.g. `cr:Field:gad7_total_score`), and group by `cr:Field:gender`.

In [ ]:
# Specify fields using their @id as per the Croissant schema
main_df = dfs[primary_table_id]

# Use the dataset structure to find actual @ids
# Example field IDs (replace with values from overview if different):
numeric_field_id = 'cr:Field:gad7_total_score'  # Total GAD-7 score
group_field_id = 'cr:Field:gender'              # Gender field @id

# Check if these fields exist
print('Available columns:', main_df.columns.tolist())

if numeric_field_id in main_df.columns:
    threshold = 10  # Moderate/severe cutoff for GAD-7
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Records with {numeric_field_id} > {threshold}: {len(filtered_df)}")
    print(filtered_df.head())

    # Normalize the scores
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field (e.g., gender)
    if group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped)
else:
    print(f"Field {numeric_field_id} not found in columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We will plot the distribution of GAD-7 scores, grouped by Gender, using matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in main_df.columns and group_field_id in main_df.columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(data=main_df, x=group_field_id, y=numeric_field_id)
    plt.title('GAD-7 Total Score by Gender')
    plt.xlabel('Gender')
    plt.ylabel('GAD-7 Total Score')
    plt.show()
else:
    print(f"Cannot plot: required fields '{numeric_field_id}' and/or '{group_field_id}' not found.")

## 6. Conclusion
This notebook demonstrates how to use the `mlcroissant` package to:
- Load dataset metadata and records referenced by their Croissant `@id`
- Explore fields and structure for each record set
- Extract tabular data for analysis and visualization
- Perform basic EDA, normalization, and grouping

For further analysis, consult the dataset's metadata for field definitions and consider examining additional record sets (demographics, responses, etc.) and documented limitations, especially around sensitive information.